# 1.1 Explore one brain MRI and Audit the Data

**Goal:** understand the image array, choose one MRI sequence, and overlay an expert annotation.

No model or accelerator is needed. Use a CPU runtime in Colab.

An MRI is a grid of **voxels**, or 3D pixels. This dataset bundles four sequences (different views of tissue contrast) into a fourth array dimension. A segmentation mask assigns an integer region label to each spatial voxel.

Source: [Medical Segmentation Decathlon on AWS](https://registry.opendata.aws/msd/). Data license: CC BY-SA 4.0. Dataset data is downloaded locally and is not included in this notebook.


# A: Inspect data examples

## 1. Prepare environment

In [ ]:
%pip install -q "boto3>=1.34,<2" "nibabel>=5.2,<6" "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
from pathlib import Path

if Path('/content').is_dir():
    DATA_DIR = Path('/content/brain_mri_sample')
else:
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parent
    DATA_DIR = project_root / 'data' / 'sample'

print('Local data folder:', DATA_DIR)


## 2. Download one case

The following utility also lives in `scripts/download_sample.py`. It lists available images, finds a matching mask, and downloads only that pair plus metadata. Existing files with the expected size are skipped.

This is ordinary Python data access; you do not need an AWS account. The cell reports the transfer size.

In [ ]:
import io
import json
import tarfile
from pathlib import Path
from urllib.request import Request, urlopen

ARCHIVE_URL = (
    "https://msd-for-monai.s3.us-west-2.amazonaws.com/"
    "Task01_BrainTumour.tar"
)


class RemoteArchive(io.RawIOBase):
    """Read selected bytes from an online archive."""

    def __init__(self, url):
        self.url = url
        self.position = 0
        self.transferred = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.position

    def seek(self, offset, whence=0):
        # Moving the position does not download the skipped bytes.
        if whence == 0:
            self.position = offset
        elif whence == 1:
            self.position += offset
        else:
            raise ValueError("Seeking from the archive end is unsupported.")

        return self.position

    def read(self, size=-1):
        if size < 0:
            raise ValueError("Unbounded downloads are disabled.")
        if size == 0:
            return b""

        start = self.position
        end = start + size - 1

        request = Request(
            self.url,
            headers={"Range": f"bytes={start}-{end}"},
        )

        with urlopen(request, timeout=60) as response:
            # Stop if the server tries to send the whole archive.
            if response.status != 206:
                raise RuntimeError(
                    "Server did not honor the partial-download request."
                )

            content_range = response.headers.get("Content-Range", "")
            if not content_range.startswith(f"bytes {start}-"):
                raise RuntimeError("Server returned an unexpected byte range.")

            data = response.read(size)

        self.position += len(data)
        self.transferred += len(data)
        return data


def download_sample(output_dir):
    output_dir = Path(output_dir)
    prefix = "Task01_BrainTumour/"
    manifest_path = output_dir / "sample_manifest.json"

    # Skip downloading if this version already saved a complete sample.
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text())

        if previous.get("source_url") == ARCHIVE_URL:
            complete = all(
                (output_dir / item["path"]).is_file()
                and (output_dir / item["path"]).stat().st_size == item["bytes"]
                for item in previous["objects"]
            )

            if complete:
                print("Sample already downloaded:", previous["case"])
                return output_dir

    selected_case = None
    saved_files = {}
    remote = RemoteArchive(ARCHIVE_URL)

    print("Scanning archive headers. This may take several minutes.")
    print("Unrelated MRI contents will be skipped.", flush=True)

    with tarfile.open(fileobj=remote, mode="r:") as archive:
        for count, member in enumerate(archive):
            if count % 100 == 0:
                print(
                    f"Inspected {count} entries; "
                    f"transferred {remote.transferred / 1_000_000:.1f} MB",
                    flush=True,
                )

            if not member.isfile():
                continue

            name = member.name.removeprefix("./").removeprefix(prefix)
            path = Path(name)

            # Ignore hidden macOS metadata files.
            if path.name.startswith("._"):
                continue

            # Choose the first MRI or mask case encountered.
            if (
                selected_case is None
                and path.parent.as_posix() in ("imagesTr", "labelsTr")
                and name.endswith(".nii.gz")
            ):
                selected_case = path.name
                print("Selected case:", selected_case, flush=True)

            wanted = {
                "dataset.json",
                f"imagesTr/{selected_case}",
                f"labelsTr/{selected_case}",
            }

            if name not in wanted or name in saved_files:
                continue

            destination = output_dir / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_name(destination.name + ".part")

            # Copy only this selected file, in manageable chunks.
            with archive.extractfile(member) as source:
                with temporary.open("wb") as target:
                    while chunk := source.read(4 * 1024 * 1024):
                        target.write(chunk)

            if temporary.stat().st_size != member.size:
                raise RuntimeError(f"Incomplete download: {name}")

            temporary.replace(destination)
            saved_files[name] = {"path": name, "bytes": member.size}
            print("Saved:", destination, flush=True)

            if len(saved_files) == 3:
                break

    if len(saved_files) != 3:
        raise RuntimeError(
            f"Could not find a complete MRI/mask pair for {selected_case}."
        )

    manifest = {
        "source_url": ARCHIVE_URL,
        "case": selected_case,
        "objects": list(saved_files.values()),
        "transferred_bytes": remote.transferred,
    }

    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

    print(f"\nDone! Transferred {remote.transferred / 1_000_000:.1f} MB.")
    return output_dir


download_sample(DATA_DIR)

In [ ]:
# Map dataset structure
from pathlib import Path


def show_folder_tree(root, max_files=3, max_depth=3):
    root = Path(root)

    if not root.is_dir():
        print(f"Folder not found: {root}")
        return

    print(f"{root.resolve()}/")

    def walk(folder, prefix="", depth=0):
        if depth >= max_depth:
            return

        items = sorted(
            folder.iterdir(),
            key=lambda p: (not p.is_dir(), p.name.lower()),
        )

        # Hide temporary and system files.
        items = [p for p in items if not p.name.startswith(".")]

        folders = [p for p in items if p.is_dir()]
        files = [p for p in items if p.is_file()]

        entries = folders + files[:max_files]
        hidden_count = max(0, len(files) - max_files)

        for index, item in enumerate(entries):
            is_last = index == len(entries) - 1 and hidden_count == 0
            branch = "└── " if is_last else "├── "

            if item.is_dir():
                print(f"{prefix}{branch}{item.name}/")
                extension = "    " if is_last else "│   "
                walk(item, prefix + extension, depth + 1)
            else:
                size_mb = item.stat().st_size / 1_000_000
                print(f"{prefix}{branch}{item.name} ({size_mb:.2f} MB)")

        if hidden_count:
            print(f"{prefix}└── ... {hidden_count} more files")

    walk(root)


show_folder_tree(DATA_DIR)

## 3. Read the sequence and annotation definitions

Do not guess what a channel or a label number means: read the dataset description. Region labels describe tumor-related tissue regions, not separate disease diagnoses.


In [ ]:
import json

metadata = json.loads((DATA_DIR / 'dataset.json').read_text())
print('MRI channels:', metadata['modality'])
print('Mask labels:', metadata['labels'])
manifest = json.loads((DATA_DIR / 'sample_manifest.json').read_text())
case_name = manifest['case']
print('Case:', case_name)


## 4. Load the MRI and mask

We arrange each file into a standard anatomical orientation, then check that their spatial grids match. The first three dimensions describe space; the fourth image dimension indexes sequences.


In [ ]:
import nibabel as nib
import numpy as np

image = nib.as_closest_canonical(nib.load(DATA_DIR / 'imagesTr' / case_name))
mask_image = nib.as_closest_canonical(nib.load(DATA_DIR / 'labelsTr' / case_name))
volume = image.get_fdata(dtype=np.float32)
mask = mask_image.get_fdata(dtype=np.float32)

assert volume.ndim == 4, f'Expected multiple sequences, got {volume.shape}'
assert mask.ndim == 3, f'Expected a 3D mask, got {mask.shape}'
assert volume.shape[:3] == mask.shape, 'Spatial shapes differ'
assert np.allclose(image.affine, mask_image.affine), 'Spatial coordinates differ'
assert np.isfinite(mask).all() and np.allclose(mask, np.round(mask)), 'Invalid mask labels'

mask = np.round(mask).astype(np.int16)
print('MRI shape:', volume.shape)
print('Mask shape:', mask.shape)
print('Voxel spacing (mm):', image.header.get_zooms()[:3])
print('Orientation:', nib.aff2axcodes(image.affine))
print('Labels present:', np.unique(mask))


## 5. Choose a sequence and a slice containing annotations

Start with channel 0 and look up its name above. The slice with the most annotated pixels is useful for this first visual check; this selection is not a model prediction.

The display uses percentile contrast limits without modifying the MRI values. The mask overlay shows all nonzero annotation regions together.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

channel = 0
scan = volume[..., channel]

# Select the slice containing the most annotated pixels.
z = int((mask > 0).sum(axis=(0, 1)).argmax())

# Adjust MRI display contrast.
finite = scan[np.isfinite(scan) & (scan != 0)]
vmin, vmax = np.percentile(finite, [1, 99])

# Fixed mapping: 0 = transparent, 1 = blue, 2 = yellow, 3 = red.
colors = ["none", "blue", "yellow", "red"]
label_cmap = ListedColormap(colors)
label_cmap.set_bad((0, 0, 0, 0))

# Keep colors fixed even when some labels are absent.
label_norm = BoundaryNorm(
    [-0.5, 0.5, 1.5, 2.5, 3.5],
    label_cmap.N,
)

slice_mask = mask[:, :, z].T
overlay = np.ma.masked_where(slice_mask == 0, slice_mask)

sequence = metadata["modality"][str(channel)]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax in axes:
    ax.imshow(
        scan[:, :, z].T,
        cmap="gray",
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    ax.axis("off")

axes[0].set_title(f"{sequence}: slice {z}")

axes[1].imshow(
    overlay,
    cmap=label_cmap,
    norm=label_norm,
    alpha=0.55,
    origin="lower",
    interpolation="nearest",
)
axes[1].set_title("Expert annotations")

legend = [
    Patch(
        facecolor=colors[label],
        label=f"{label}: {metadata['labels'][str(label)]}",
    )
    for label in (1, 2, 3)
]

fig.legend(
    handles=legend,
    loc="lower center",
    ncol=3,
    frameon=False,
)

plt.tight_layout(rect=(0, 0.10, 1, 0.95))
plt.show()

In [ ]:
print("MRI shape:", volume.shape)
print("Mask shape:", mask.shape)
print("Region meanings:", metadata["labels"])

# B: Audit starter data

Objectives:
- Check what the data can teach a model.
- Check files, annotations, and missing information before training.

## 1. Check paired volume-mask files and count annotation values

In [ ]:
images = {p.name: p for p in (DATA_DIR / 'imagesTr').glob('*.nii.gz') if not p.name.startswith('._')}
masks = {p.name: p for p in (DATA_DIR / 'labelsTr').glob('*.nii.gz') if not p.name.startswith('._')}
assert images, 'No MRI files found.'
assert images.keys() == masks.keys(), 'Some images or masks have no matching file.'
print(f'Paired cases downloaded: {len(images)}')

records = []
for name in sorted(images):
    image = nib.as_closest_canonical(nib.load(images[name]))
    label_image = nib.as_closest_canonical(nib.load(masks[name]))
    labels = label_image.get_fdata(dtype=np.float32)
    assert len(image.shape) == 4 and labels.ndim == 3, 'Unexpected array dimensions'
    assert image.shape[:3] == labels.shape, 'Spatial shapes differ'
    assert np.allclose(image.affine, label_image.affine), 'Spatial grids differ'
    assert np.isfinite(labels).all() and np.allclose(labels, np.round(labels)), 'Invalid mask values'
    values, counts = np.unique(labels.astype(np.int16), return_counts=True)
    known = {int(key) for key in metadata['labels']}
    assert set(values).issubset(known), 'Unexpected annotation label'
    spacing = image.header.get_zooms()[:3]
    voxel_ml = float(np.prod(spacing)) / 1000
    print(f'\n{name}: shape={image.shape}, spacing={spacing} mm')
    region_counts = {str(int(value)): int(count) for value, count in zip(values, counts)}
    for value, count in zip(values, counts):
        meaning = metadata['labels'][str(int(value))]
        print(f'  {int(value)}: {meaning}: {count:,} voxels ({count * voxel_ml:.2f} mL)')
    records.append({
        'case_file': name,
        'shape': list(image.shape),
        'spacing_mm': [float(v) for v in spacing],
        'region_voxel_counts': region_counts,
        'patient_identity_verified': False,
        'diagnostic_labels': None,
        'split': 'unassigned',
    })
print(f"\nCase: {name}")
print("MRI shape:         ", image.shape)
print("MRI spatial shape: ", image.shape[:3])
print("Mask shape:        ", labels.shape)

sizes_match = image.shape[:3] == labels.shape
print("Spatial sizes match:", sizes_match)

assert sizes_match, "Spatial shapes differ"


💭 Comments:
1. The mask applies to the correct corresponding raw MRI scan.
2. 47 voxels were found for non-enhancing tumor when searching ACROSS 155 slices (max depth).

## 2. Download the example case

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/msd_brain_tumour_sample",
    "zip",
    root_dir=str(DATA_DIR),
)

files.download(zip_path)

## 3. Key takeaways about Dataset: MSD Task01 — Brain Tumour.

- `.nii.gz` is a compressed NIfTI file containing image data and metadata such as voxel spacing and orientation.
- `imagesTr/` contains MRI scans; `labelsTr/` contains their matching expert segmentation masks.
- `BRATS_457` identifies a case, not a slice or sequence. Matching image and mask files share this name.
- The MRI shape (240, 240, 155, 4) represents three spatial dimensions and four MRI sequences of the same brain.
- A volume is a 3D image; a slice is a 2D cross-section; a voxel is a 3D pixel. Different sequences show tissue with different contrast.
- The mask shape (240, 240, 155) matches the MRI’s spatial grid and assigns one label per voxel:
  - 0: background—outside annotated regions.
  - 1: edema—displayed blue.
  - 2: non-enhancing tumor—displayed yellow.
  - 3: enhancing tumor—displayed red.

**Note**:
- These colors are chosen by our plotting code; they do not indicate severity.
- These labels describe tumor-related regions, not tumor diagnoses. We check both spatial shape and coordinate metadata before overlaying the mask on the MRI.